In [4]:
import json
import re
from pathlib import Path
import pandas as pd

In [10]:
m3cot_ivtlr = Path("/home/csalt/Haider/DVLM/IVT-LR/qwen_vl/output/inference/m3cot")
sqa_ivtlr = Path("/home/csalt/Haider/DVLM/IVT-LR/qwen_vl/output/inference/sqa")
baseline = Path("/home/csalt/Haider/DVLM/lvar/outputs/baseline_and_pooling_results")

In [5]:
def _parse_final_line(log_path: Path):
    if not log_path or not log_path.exists():
        return None
    final_pattern = re.compile(
        r"\[FINAL\]\s*Total:\s*(\d+),\s*Correct:\s*(\d+),\s*Accuracy:\s*([0-9.]+)%"
    )
    # Read last 200 lines to capture the final summary without loading huge logs.
    try:
        lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    except OSError:
        return None
    for line in reversed(lines[-200:]):
        match = final_pattern.search(line)
        if match:
            total = int(match.group(1))
            correct = int(match.group(2))
            acc = float(match.group(3))
            return {"total": total, "correct": correct, "accuracy": acc}
    return None


def _pick_log(epoch_dir: Path, pattern: str):
    matches = sorted(epoch_dir.glob(pattern))
    if matches:
        return matches[0]
    # Fallback: any stdout/stderr log file in the folder.
    fallback = sorted(epoch_dir.glob("*stdout*stderr*.log"))
    return fallback[0] if fallback else None


def _format_cell(metrics):
    if not metrics:
        return ""
    return f"{metrics['correct']}/{metrics['total']} ({metrics['accuracy']:.2f}%)"


In [ ]:
rows = []

for epoch_dir in sorted(m3cot_ivtlr.glob("epoch_*")):
    log_path = _pick_log(epoch_dir, "*m3cot*stdout*stderr*.log")
    metrics = _parse_final_line(log_path)
    rows.append({
        "Model Type": epoch_dir.name,
        "m3cot": metrics,
        "scienceqa": None,
        "clevr": None,
    })

for epoch_dir in sorted(sqa_ivtlr.glob("epoch_*")):
    log_path = _pick_log(epoch_dir, "*sqa*stdout*stderr*.log")
    metrics = _parse_final_line(log_path)
    rows.append({
        "Model Type": epoch_dir.name,
        "m3cot": None,
        "scienceqa": metrics,
        "clevr": None,
    })

# Merge rows by epoch name so both datasets land on the same row.
merged = {}
for row in rows:
    key = row["Model Type"]
    merged.setdefault(key, {"Model Type": key, "m3cot": None, "scienceqa": None, "clevr": None})
    for col in ["m3cot", "scienceqa", "clevr"]:
        if row[col] is not None:
            merged[key][col] = row[col]

rows = [merged[k] for k in sorted(merged.keys(), key=lambda x: int(x.split("_")[-1]))]

baseline_files = sorted(baseline.glob("*pooling_predictions_summary.json"))
baseline_rows = {}
for summary_path in baseline_files:
    with summary_path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    metrics = data.get("metrics", {})
    dataset_name = summary_path.name.replace("_pooling_predictions_summary.json", "")
    for key in [
        "full_image",
        "mean_pooled",
        "max_pooled",
        "region_mean_pooled",
        "region_max_pooled",
    ]:
        entry = metrics.get(key)
        if not entry:
            continue
        row_name = f"baseline_{key}"
        row = baseline_rows.setdefault(
            row_name,
            {"Model Type": row_name, "m3cot": None, "scienceqa": None, "clevr": None},
        )
        normalized = {
            "correct": entry.get("correct"),
            "total": entry.get("total"),
            "accuracy": entry.get("accuracy", 0.0) * 100.0,
        }
        if dataset_name == "m3cot":
            row["m3cot"] = normalized
        elif dataset_name in {"scienceqa", "sqa"}:
            row["scienceqa"] = normalized
        elif dataset_name == "clevr":
            row["clevr"] = normalized

rows.extend([baseline_rows[k] for k in sorted(baseline_rows.keys())])

In [30]:

def _metric_to_count(metrics):
    if not metrics:
        return ""
    return f"{metrics['correct']}/{metrics['total']}"


def _metric_to_accuracy(metrics):
    if not metrics:
        return ""
    return f"{metrics['accuracy']:.2f}%"


def _metric_to_both(metrics):
    if not metrics:
        return ""
    return f"{metrics['correct']}/{metrics['total']} ({metrics['accuracy']:.2f}%)"


def _render_df(raw_rows, value_fn):
    rendered = []
    for raw in raw_rows:
        rendered.append(
            {
                "Model Type": raw["Model Type"],
                "m3cot": value_fn(raw.get("m3cot")),
                "scienceqa": value_fn(raw.get("scienceqa")),
                "clevr": value_fn(raw.get("clevr")),
            }
        )
    return pd.DataFrame(rendered).set_index("Model Type")


column_map = {
    "m3cot": "M3Cot",
    "scienceqa": "ScienceQA",
    "clevr": "Clevr",
}

baseline_map = {
    "baseline_full_image": "Baseline (Full Image)",
    "baseline_mean_pooled": "Baseline (Mean Pooled)",
    "baseline_max_pooled": "Baseline (Max Pooled)",
    "baseline_region_mean_pooled": "Baseline (Region Mean Pooled)",
    "baseline_region_max_pooled": "Baseline (Region Max Pooled)",
}


def _apply_labels(df):
    df = df.rename(columns=column_map)
    df.index = df.index.map(
        lambda name: baseline_map.get(
            name,
            name.replace("epoch_", "Epoch ") if name.startswith("epoch_") else name,
        )
    )
    return df


table_both = _apply_labels(_render_df(rows, _metric_to_both))
table_counts = _apply_labels(_render_df(rows, _metric_to_count))
table_accuracy = _apply_labels(_render_df(rows, _metric_to_accuracy))

In [31]:
display(table_accuracy)
display(table_counts)
display(table_both)

,M3Cot,ScienceQA,Clevr
Model Type,,,
Epoch 4,33.61%,46.21%,
Epoch 8,56.04%,79.33%,
Epoch 12,55.95%,86.66%,
Epoch 16,64.75%,92.27%,
Baseline (Full Image),43.01%,76.90%,95.06%
Baseline (Max Pooled),34.69%,64.60%,33.30%
Baseline (Mean Pooled),35.38%,65.94%,32.39%
Baseline (Region Max Pooled),39.00%,73.48%,63.26%
Baseline (Region Mean Pooled),41.50%,75.51%,75.87%


,M3Cot,ScienceQA,Clevr
Model Type,,,
Epoch 4,779/2318,932/2017,
Epoch 8,1299/2318,1600/2017,
Epoch 12,1297/2318,1748/2017,
Epoch 16,1501/2318,1861/2017,
Baseline (Full Image),997/2318,1551/2017,6654/7000
Baseline (Max Pooled),804/2318,1303/2017,2331/7000
Baseline (Mean Pooled),820/2318,1330/2017,2267/7000
Baseline (Region Max Pooled),904/2318,1482/2017,4428/7000
Baseline (Region Mean Pooled),962/2318,1523/2017,5311/7000


,M3Cot,ScienceQA,Clevr
Model Type,,,
Epoch 4,779/2318 (33.61%),932/2017 (46.21%),
Epoch 8,1299/2318 (56.04%),1600/2017 (79.33%),
Epoch 12,1297/2318 (55.95%),1748/2017 (86.66%),
Epoch 16,1501/2318 (64.75%),1861/2017 (92.27%),
Baseline (Full Image),997/2318 (43.01%),1551/2017 (76.90%),6654/7000 (95.06%)
Baseline (Max Pooled),804/2318 (34.69%),1303/2017 (64.60%),2331/7000 (33.30%)
Baseline (Mean Pooled),820/2318 (35.38%),1330/2017 (65.94%),2267/7000 (32.39%)
Baseline (Region Max Pooled),904/2318 (39.00%),1482/2017 (73.48%),4428/7000 (63.26%)
Baseline (Region Mean Pooled),962/2318 (41.50%),1523/2017 (75.51%),5311/7000 (75.87%)
